# Evaluación de la calidad de los datos

Comenzaremos cuantificando el impacto de los valores faltantes y realizando un análisis de frecuencia en cantidades negativas para confirmar la hipótesis de "cancelación". Esto nos ayudará a comprender el alcance de los problemas de calidad de los datos y guiará nuestros esfuerzos de limpieza y preparación de datos.

In [228]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv("../../data/raw/online_retail_II.csv" , encoding="ISO-8859-1")

Vamos a convertir la columna de fecha de factura a un formato de fecha y hora y comprobaremos si faltan valores en el conjunto de datos. Esto nos permitirá identificar cualquier problema potencial con los datos y garantizar que tengamos un conjunto de datos completo y preciso para nuestro análisis.

In [229]:
df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    format="%m/%d/%y %H:%M",
    errors="coerce"
)

In [230]:
df.isna().sum() / len(df) * 100

Invoice         0.000000
StockCode       0.000000
Description     0.268310
Quantity        0.000000
InvoiceDate     0.000000
Price           0.000000
Customer ID    24.926648
Country         0.000000
dtype: float64

Podemos analizar que los valores faltantes en el conjunto de datos son mínimos, con solo un porcentaje alto en ID de cliente y porcentajes más bajos en Descripción. Por lo tanto, podemos considerar reemplazar Descripción con un valor de marcador de posición como "Unknown" o "No Description" para mantener la integridad del conjunto de datos. Para el ID de cliente, podemos considerar eliminar esas filas o imputarlas en función de otra información disponible.

Mientras tanto, tenemos que investigar por qué hay un 24,92% en ID de cliente. Esto podría estar relacionado con la naturaleza del conjunto de datos, donde algunas transacciones pueden no tener información del cliente asociada. Podemos explorar esto más a fondo analizando la distribución de la ID del cliente y verificando cualquier patrón o correlación con otras variables.

In [231]:
df[df['Customer ID'].isna()].head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.00,NaN,United Kingdom
1443,536544,21773,DECORATIVE ROSE BATHROOM BOTTLE,1,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1444,536544,21774,DECORATIVE CATS BATHROOM BOTTLE,2,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1445,536544,21786,POLKADOT RAIN HAT,4,2010-12-01 14:32:00,0.85,NaN,United Kingdom
1446,536544,21787,RAIN PONCHO RETROSPOT,2,2010-12-01 14:32:00,1.66,NaN,United Kingdom


### ¿Qué hacemos con los valores faltantes del ID de cliente?

El propósito de este análisis es comprender los segmentos de clientes y su comportamiento de compra. Dado que el ID de cliente es un identificador crucial para segmentar clientes, debemos considerar cuidadosamente cómo manejar los valores faltantes.

Por lo tanto, mantendremos las filas a las que les falta el ID de cliente por ahora, pero cuando realicemos el análisis de segmentación de clientes, excluiremos esas filas del análisis. De esta manera, aún podemos analizar las transacciones y sus características sin perder información valiosa.

### ¿Qué hacemos con los valores faltantes de Descripción?

Si bien los valores que faltan en Descripción son relativamente bajos, podemos reemplazarlos con un valor de marcador de posición como "Desconocido" o "Sin descripción" para mantener la integridad del conjunto de datos. Esto nos permitirá conservar esas filas para su análisis sin perder ninguna información valiosa.

In [232]:
print(f"Number of duplicated rows: {df.duplicated().sum()}")
df.apply(lambda col: col.value_counts().gt(1).sum())

Number of duplicated rows: 5268


Invoice        20059
StockCode       3837
Description     3915
Quantity         414
InvoiceDate    19018
Price            526
Customer ID     4293
Country           38
dtype: int64

In [233]:
duplicate_rows = df[
    df.duplicated(
        subset=[
            'Invoice',
            'StockCode',
            'Description',
            'Price',
            'Customer ID',
            'Quantity',
        ],
        keep=False,
    )
]

print(f"Number of duplicated rows: {len(duplicate_rows)}")
print(f"Percentage of duplicated rows: {len(duplicate_rows) / len(df) * 100:.2f}%")

Number of duplicated rows: 10149
Percentage of duplicated rows: 1.87%


Tenemos 10149 filas duplicadas en el conjunto de datos, lo que representa solo el 1,87% del conjunto de datos total. Podemos considerar eliminar esas filas para mantener la integridad del conjunto de datos y evitar cualquier posible sesgo en nuestro análisis. Al eliminar duplicados, podemos garantizar que nuestro análisis se base en transacciones e interacciones únicas con los clientes, lo que genera información más precisa sobre los segmentos de clientes y el comportamiento de compra.

In [234]:
df = df.drop_duplicates(
    subset=[
                'Invoice',
                'StockCode',
                'Description',
                'Price',
                'Customer ID',
                'Quantity',
            ],
    keep='first'
)

print(f"Number of duplicated rows after dropping duplicates: {df.duplicated().sum()}")
print(f"Percentage of duplicated rows after dropping duplicates: {df.duplicated().sum() / len(df) * 100:.2f}%")

Number of duplicated rows after dropping duplicates: 0
Percentage of duplicated rows after dropping duplicates: 0.00%


También podemos comprobar que los valores duplicados forman parte de la lógica de negocio, ya que están relacionados con el mismo número de factura. Por lo tanto, no los eliminaremos del conjunto de datos.

In [235]:
invalid_dates = df[
    (df["InvoiceDate"] < "2009-12-01") &
    (df["InvoiceDate"] > "2011-12-10")
]

print(f"Number of invalid dates: {len(invalid_dates)}")

Number of invalid dates: 0


No hay fechas no válidas en el conjunto de datos, ya que todas las fechas están dentro del rango esperado **(entre 2009-12-01 y 2011-12-10)**. Esto indica que la información de la fecha es confiable y se puede utilizar para análisis posteriores sin preocuparse por problemas de calidad de los datos relacionados con las fechas.

In [236]:
df[df["Quantity"] <= 0].head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


Ahora, podemos analizar por qué la cantidad es inferior a 0, lo que podría estar relacionado con la naturaleza del conjunto de datos, porque hay un código en el número de factura que comienza con "C", lo que indica una cancelación. Por lo tanto, podemos considerar esas filas como transacciones válidas y mantenerlas en el conjunto de datos para su análisis, pero luego crearemos una nueva columna para indicar si la transacción es una cancelación o no, según el número de factura. Esto nos permitirá analizar las cancelaciones por separado y comprender su impacto en los segmentos generales de clientes y el comportamiento de compra.

Además, comprobaremos si las cantidades negativas están relacionadas con los números de factura que empiezan por "C", lo que indica una cancelación. Si encontramos **algunas cantidades negativas que no están relacionadas con cancelaciones**, investigaremos más a fondo para comprender el motivo de esas transacciones y determinar si deben incluirse en el análisis o no.

Por otro lado, descubrimos un nuevo tipo de StockCode llamado "D", que está relacionado con descuentos, y nos da una cantidad en negativo, por lo que esto nos indica que las cantidades negativas no sólo están relacionadas con cancelaciones, sino también con descuentos. Por lo tanto, necesitaremos considerar tanto las cancelaciones como los descuentos al analizar las cantidades negativas en el conjunto de datos.

In [237]:
percent_cancellations = df[df["Quantity"] <= 0]["Invoice"].str.startswith("C").sum() / len(df[df["Quantity"] < 0]) * 100

print(f"Percentage of cancellations among negative quantities: {percent_cancellations:.2f}%")

Percentage of cancellations among negative quantities: 87.38%


El porcentaje de cancelaciones entre cantidades negativas con números de factura que comienzan con "C" es **87,38%**, lo que indica que la mayoría de cantidades negativas están efectivamente relacionadas con cancelaciones. Sin embargo, aún quedan algunas cantidades negativas que no están asociadas a cancelaciones, sobre las que vamos a investigar más a fondo para entender su naturaleza.

In [238]:
not_invoice_cancellations = df[(df["Quantity"] <= 0) & (~df["Invoice"].str.startswith("C"))]
print(f"Number of negative quantities not related to invoice cancellations: {len(not_invoice_cancellations)}")
print(f"Percentage of negative quantities not related with the total of invoice cancellations: {len(not_invoice_cancellations) / len(df[df['Quantity'] < 0]) * 100:.2f}%")

Number of negative quantities not related to invoice cancellations: 1336
Percentage of negative quantities not related with the total of invoice cancellations: 12.62%


En resumen, sólo el 12,62% de las cantidades negativas no están relacionadas con cancelaciones de facturas, lo que supone un porcentaje relativamente pequeño. Por lo tanto, podemos eliminar esas filas del conjunto de datos, ya que pueden no ser relevantes para nuestro análisis y podrían introducir ruido o sesgo en nuestros resultados. Al centrarnos en las transacciones válidas y cancelaciones, podemos obtener una comprensión más clara de los segmentos de clientes y su comportamiento de compra.

In [239]:
negative_prices = df[df["Price"] <= 0].head()
print(f"Number of negative prices: {len(negative_prices)}")
print(f"Percentage of negative prices: {len(negative_prices) / len(df) * 100:.5f}%")
negative_prices.head()

Number of negative prices: 5
Percentage of negative prices: 0.00093%


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1510,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1985,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1986,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
2022,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom


También podemos analizar los precios negativos en el conjunto de datos, que podemos analizar más a fondo y solo tenemos **0,00093%** del conjunto de datos. Concluimos que es probable que los precios negativos sean errores o anomalías en el ingreso de datos, y podemos considerar eliminar esas filas porque no están relacionadas con algunos códigos de acciones especiales. Por lo tanto, debemos eliminar estas filas del conjunto de datos para mantener la integridad del análisis. Al eliminar estos valores atípicos, podemos garantizar que nuestro análisis se base en datos precisos y confiables, lo que generará información más significativa sobre los segmentos de clientes y el comportamiento de compra.

In [240]:
df[(df["StockCode"].str.len() <= 4)]["StockCode"].value_counts()

StockCode
POST    1257
DOT      710
M        566
C2       144
D         77
S         62
CRUK      16
PADS       4
B          3
m          1
Name: count, dtype: int64

Descubrimos que la empresa utiliza una pequeña cantidad de StockCodes con menos de 5 caracteres, que probablemente estén relacionados con descuentos u otros casos especiales.

## Tipos

- **POST**: Envío
- **PUNTO**: Franqueo Dotcom
- **M** : Manual
- **C2** : Carro
- **D** : Descuento
- **S**: Muestras
- **CRUK**: Comisión CRUK
- **PADS**: PADS TO MATCH ALL CUSHIONS
- **B**: Ajustar deudas incobrables

Por lo tanto, podemos concluir que existen muchos tipos de códigos de acciones que nos brindan más contexto sobre la naturaleza de las transacciones y su impacto en los segmentos generales de clientes y el comportamiento de compra. Al analizar estos códigos de acciones, podemos comprender mejor los diferentes tipos de transacciones y su importancia en el conjunto de datos.

In [241]:
special_stock_codes = ["POST", "DOT", "M", "C2", "D", "S", "CRUK", "PADS", "B", "m"]

df[df["StockCode"].isin(
    special_stock_codes
)][
    ["StockCode", "Description"]
].drop_duplicates().sort_values("StockCode")

,StockCode,Description
299982,B,Adjust bad debt
1423,C2,CARRIAGE
453999,C2,NaN
317508,CRUK,CRUK Commission
141,D,Discount
1815,DOT,DOTCOM POSTAGE
136537,DOT,NaN
2239,M,Manual
157195,PADS,PADS TO MATCH ALL CUSHIONS
45,POST,POSTAGE


Podemos apreciar que algunas descripciones no están relacionadas con el código bursátil, lo que indica que existen algunas inconsistencias en el conjunto de datos. Esto podría deberse a errores en la entrada de datos u otros factores, y cambiaremos esto más adelante en el paso de preparación de datos. Al abordar estas inconsistencias, podemos garantizar que nuestro análisis se base en datos precisos y confiables, lo que generará información más significativa sobre los segmentos de clientes y el comportamiento de compra.

In [242]:
quantity_negatives_special_stock_codes =   df[(df["StockCode"].isin(special_stock_codes)) & (df["Quantity"] < 0) & ((~df["Invoice"].str.startswith("C")))]["StockCode"].value_counts().head(10)

print(f"Number of negative quantities not related to invoice cancellations and special stock codes: {len(quantity_negatives_special_stock_codes)}")

Number of negative quantities not related to invoice cancellations and special stock codes: 0


No existe relación entre los errores y este tipo de códigos bursátiles especiales, por lo que podemos concluir que las inconsistencias en el conjunto de datos probablemente se deben a errores en el ingreso de datos u otros factores, en lugar de estar relacionados con tipos específicos de códigos bursátiles.

In [243]:
df.to_csv("../../data/processed/online_retail_II_without_duplicated.csv", index=False)

## Conclusión de la evaluación de la calidad de los datos

En general, el conjunto de datos se puede utilizar para la segmentación de clientes, pero contiene varios problemas de calidad de los datos que deben abordarse antes de modelar. Los hallazgos más importantes son:

* **Valores faltantes**: Los valores faltantes están presentes, especialmente en ID de cliente (24,93%), que es un campo crítico para la segmentación. Estas filas deben excluirse del análisis a nivel de cliente y al mismo tiempo preservarse el contexto de la transacción cuando sea necesario.

* **Limpieza de la descripción**: Falta descripción en una pequeña proporción de registros (0,27%); estos se pueden reemplazar de forma segura con un marcador de posición como "Desconocido" o "Sin descripción".

* **Registros duplicados**: se identificaron registros duplicados (10149 filas, que representan el 1,87 % del conjunto de datos) y se eliminaron con éxito utilizando`drop_duplicates`para garantizar la unicidad de las transacciones y reducir el ruido sin afectar la lógica empresarial subyacente.

* **Confiabilidad temporal**: las fechas de las facturas son válidas y están dentro del rango esperado (entre el 1 de diciembre de 2009 y el 10 de diciembre de 2011), por lo que no se requiere ninguna limpieza importante relacionada con las fechas.

* **Dinámica de cancelación**: las cantidades negativas están vinculadas principalmente a cancelaciones de facturas (números de factura que comienzan con "C") y representan el 87,38 % de las cantidades negativas, que parecen ser eventos comerciales válidos en lugar de errores.

* **Negativos de no cancelación**: Una pequeña porción de cantidades negativas (12,62%) no corresponde a cancelaciones y deben revisarse o eliminarse como transacciones no válidas.

* **Anomalías de precios**: Los precios negativos o cero son insignificantes (solo 5 registros, o 0,00093%) y pueden tratarse como anomalías o eliminarse de forma segura para mantener la integridad del cálculo.

* **Códigos de acciones especiales**: Los códigos de acciones especiales como POST, DOT, M, C2, D, S, CRUK, PADS y B corresponden a ajustes, descuentos, envíos u otras transacciones no estándar. Algunos de estos códigos presentaban descripciones faltantes o desalineadas que requieren normalización durante la preparación de los datos.

En resumen, el conjunto de datos no está lo suficientemente sucio como para invalidar el análisis, pero sí requiere una limpieza específica y un tratamiento cuidadoso de las cancelaciones, los códigos de stock no estándar y la información faltante de los clientes. Una vez que se aborden estos problemas, el conjunto de datos será adecuado para un flujo de trabajo confiable de segmentación de clientes.

---

## Siguiente paso: preparación de datos

La siguiente fase, Preparación de datos, se centrará en transformar el conjunto de datos en una estructura limpia y lista para el análisis para la segmentación. El camino propuesto es:

* **Estandarizar columnas y tipos de datos**
    * Convierta fechas al formato de fecha y hora utilizando el diseño de análisis correcto.
    * Asegúrese de que los campos numéricos sean numéricos.
    * Limpie los nombres de las columnas para mantener la coherencia.
* **Manejar valores faltantes**
    * Reemplace los valores de descripción que faltan con "Desconocido".
    * Eliminar o marcar filas en las que falta el ID de cliente para el análisis a nivel de cliente.
* **Eliminar transacciones no válidas o no relevantes**
    * Elimine filas duplicadas según subconjuntos de transacciones únicos.
    * Eliminar registros de precios cero o negativos.
    * Excluye filas de cantidades negativas que no sean de cancelación a menos que sean explícitamente válidas.
* **Crear banderas comerciales**
    * Agregue el indicador `is_cancellation` según el prefijo de factura "C".
    * Agregue la bandera `special_stock_code` para ajustes, descuentos, envíos y muestras.
    * Agregue un indicador `valid_transaction` para los registros retenidos en el análisis.
* **Normalizar y enriquecer el conjunto de datos**
    * Estandarizar valores de StockCode y Descripción.
    * Manejar casos especiales como descuentos y envíos de manera consistente.
    * Obtenga características útiles a nivel de transacción, como ventas totales, valor del pedido e indicadores de compra reciente.
* **Crear un conjunto de datos a nivel de cliente**
    * Filtrar por transacciones válidas de clientes.
    * Agregado por ID de cliente.
    * Cree funciones de segmentación como gasto total, número de pedidos, valor promedio de los pedidos, actualidad, frecuencia y diversidad de productos.
* **Guardar el conjunto de datos preparado**
    * Exportar un conjunto de datos de transacciones limpio.
    * Exporte un conjunto de datos a nivel de cliente listo para agrupación y segmentación.

Esta etapa de preparación garantizará que los modelos de segmentación se basen en datos confiables, consistentes y significativos para el negocio.